In [4]:
import json
import os
import pandas as pd
import glob
from transformers import AutoTokenizer
import re
import json
from dotenv import load_dotenv
from copy import deepcopy
load_dotenv()
os.chdir(os.getenv('PARENT_DIR'))

In [5]:
os.environ['CUDA_VISIBLE_DEVICES'] = '7'

In [6]:
from typing import List, Dict, Any, Literal

In [7]:
def parse_absa_string(text: str):
	"""
	Parses a string formatted as "[A] aspect [O] opinion [S] sentiment" into a list of dictionaries.
	Each dictionary contains the tag as the key and the corresponding value.
	For example, "[A] [O] [S] [A] harga [O] terjangkau [S] positive [SSEP] [A] fasilitas [O] nyaman [S] positive" becomes:
	[{'A': 'harga', 'S': 'positive', 'O': 'terjangkau'},
	{'A': 'fasilitas', 'S': 'positive', 'O': 'nyaman'}].

	Args:
		text (str): ABSA string output to be parsed.

	Returns:
		List[Dict[str, str]]: List of dictionaries of parsed ABSA output.

	"""
	pattern = r"\[(\w+)\]\s*([^[]+)"
	matches = re.findall(pattern, text)

	result = []
	current_dict = {}

	for tag, content in matches:
		if tag == "SSEP":  # Sentence separator -> Start a new dictionary
			result.append(current_dict)
			current_dict = {}
		else:
			current_dict[tag] = content.strip()

	if current_dict:  # Append the last sentence if it exists
		result.append(current_dict)

	return result

def convert_to_absa_format(triplets: List[Dict[str, str]], order: List[Literal['A', 'O', 'S']]) -> str:
	"""
	Converts a list of dictionaries containing ABSA triplets into a formatted string.
	Each dictionary should contain keys 'A', 'O', and 'S' for Aspect, Opinion, and Sentiment respectively.
	The order of these elements in the output string is determined by the 'order' parameter.

	Args:
		triplets (List[Dict[str, str]]): List of dictionaries with ABSA triplet information.
	Returns:
		str: A formatted string representing the ABSA triplets.
	"""
	result = []
	for triplet in triplets:
		parts = []
		for key in order:
			if key in triplet:
				parts.append(f"[{key}] {triplet[key]}")
		result.append(" ".join(parts))
	return " [SSEP] ".join(result)

## Construct from MvP format

### GAS

In [11]:
lang = 'indo'
original_order = 'aos'
dataset_type = f'hoasa_hotel{"_" if (original_order != "aos" and original_order) else ""}{original_order if original_order != "aos" else ""}'
dataset_folder = 'mvp_aos'
dataset_per_split = {}
splits = ['train', 'dev', 'test']
dataset_type

'hoasa_hotel'

In [12]:
# Read the modified files and combine them into train, dev, test splits
for split in splits:
	dataset_per_split[split] = []
	files = glob.glob(f'dataset/{dataset_type}/{lang}/{dataset_folder}/*{split}*.json')
	print("Reading files for", split, ":", files)
	for file in files:
		with open(file, 'r') as f:
			data = json.load(f)
		dataset_per_split[split].extend(data)

Reading files for train : ['dataset/hoasa_hotel/indo/mvp_aos/train.json']
Reading files for dev : ['dataset/hoasa_hotel/indo/mvp_aos/dev.json']
Reading files for test : ['dataset/hoasa_hotel/indo/mvp_aos/test.json']


In [13]:
def convert_to_gas_format(data_list, order='aos'):
	"""
	Converts a list of aspect-based sentiment dictionaries to the 
	extraction-style string format used in the GAS paper.

	Args:
		data_list: A list of dictionaries, where each dictionary must contain 
				the keys 'A' (Aspect), 'O' (Opinion), and 'S' (Sentiment).

	Returns:
		A single string formatted as '(A | O | S) ; (A | O | S) ; ...'
	"""
	# A list comprehension to create a formatted string for each dictionary
	# The order of elements in the tuple is specified as A, O, S
	listed_order = list(order.upper())
	items = []
	for item in data_list:
		parts = []
		for key in listed_order:
			parts.append(item[key])
		items.append(f"( {' | '.join(parts)} )")

	# Join the list of strings together with a semicolon and space
	return " ; ".join(items)

In [14]:
gas_data_per_split = {}
for split in splits:
	gas_data_per_split[split] = []
	instance_id_counter = dataset_per_split[split][0]['sentence_id']
	for instance in dataset_per_split[split]:
		absa_string = instance['target']
		parsed_absa = parse_absa_string(absa_string)
		try:
			listed_order = list(instance['element_order'].upper())
			for d in parsed_absa:
				assert all(key in d.keys() for key in listed_order)
		except:
			print("Error in instance:", instance)
			raise ValueError("Parsed ABSA contains invalid keys.")
		gas_format = convert_to_gas_format(parsed_absa, order='aso')# Change order if needed
		suffix = ' '.join(f'[{key}]' for key in listed_order)
		gas_data_per_split[split].append({
			"sentence_id": instance['sentence_id'],
			"instance_id": instance_id_counter,
			'task_elements': instance['task_elements'],
			"input": f"{instance['input'].replace(suffix, '').strip()}", # Add arrow for input target delimiter (remove if not needed)
			"target": gas_format.strip(),
			"element_order": instance['element_order'],
			"dataset_type": instance['dataset_type']
		})
		instance_id_counter += 1

In [15]:
len(gas_data_per_split['train']), len(gas_data_per_split['dev']), len(gas_data_per_split['test'])

(4764, 1284, 1286)

In [16]:
for split in splits:
    gas_dataset_path = f'dataset/{dataset_type}/{lang}/gas_aso/{split}.json'
    os.makedirs(os.path.dirname(gas_dataset_path), exist_ok=True)
    with open(gas_dataset_path, 'w') as f:
        json.dump(gas_data_per_split[split], f, ensure_ascii=False, indent=4)
    print(f"GAS formatted dataset saved to {gas_dataset_path}")

GAS formatted dataset saved to dataset/hoasa_hotel/indo/gas_aso/train.json
GAS formatted dataset saved to dataset/hoasa_hotel/indo/gas_aso/dev.json
GAS formatted dataset saved to dataset/hoasa_hotel/indo/gas_aso/test.json


### Lego-ABSA

In [121]:
lang = 'indo'
original_order = 'o'
dataset_type = f'hoasa_hotel{"_" if (original_order != "aos" and original_order) else ""}{original_order if original_order != "aos" else ""}'
dataset_folder = 'mvp_aos'
dataset_per_split = {}
splits = ['train', 'dev', 'test']
dataset_type

'hoasa_hotel_o'

In [122]:
# Read the modified files and combine them into train, dev, test splits
for split in splits:
	dataset_per_split[split] = []
	files = glob.glob(f'dataset/{dataset_type}/{lang}/{dataset_folder}/*{split}*.json')
	print("Reading files for", split, ":", files)
	for file in files:
		with open(file, 'r') as f:
			data = json.load(f)
		dataset_per_split[split].extend(data)

Reading files for train : ['dataset/hoasa_hotel_o/indo/mvp_aos/train.json']
Reading files for dev : ['dataset/hoasa_hotel_o/indo/mvp_aos/dev.json']
Reading files for test : ['dataset/hoasa_hotel_o/indo/mvp_aos/test.json']


In [123]:
from itertools import combinations
subsets = []
for i in range(1, len(original_order) + 1):
	subsets.extend(combinations(original_order, i))

# Reverse the subset_str list to have larger subsets first
subset_str_list = [''.join(subset) for subset in subsets][::-1]

# Remove 's' from the list
# subset_str_list.remove('s')
subset_str_list

['o']

In [125]:
element_order_dict = {
    # 'indolegoabsa_multitask': ['aos', 'ao', 'as', 'a', 's'],
    # 'indolegoabsa_multitask': ['ao', 'a', 'o'],
    # 'indolegoabsa_multitask': ['as', 'a'],
    # 'indolegoabsa_multitask': ['a'],
    'indolegoabsa_multitask': ['o'],
    # 'legoabsa_multitask': ['aos', 'ao', 'as'],
    # 'legoabsa_tasktransfer': ['oa', 'as']
}
# element_order_dict = {
#     'indolegoabsa_multitask': subset_str_list,
# }

In [126]:
subtasks_orders_per_dataset = {k: [list(order.upper()) for order in v] for k, v in element_order_dict.items()}
subtasks_orders_per_dataset

{'indolegoabsa_multitask': [['O']]}

In [127]:
special_tokens = {
	'a': '<|aspect|>',
	'o': '<|opinion|>',
	's': '<|sentiment|>'
}

initial_definitions = {
    'a': 'aspect',
    'o': 'opinion',
    's': 'sentiment'
}

In [128]:
def convert_output_to_legoabsa_format(data_list, order='aos'):
	"""
	Converts a list of aspect-based sentiment dictionaries to the 
	extraction-style string format used in the Lego-ABSA paper.

	Args:
		data_list: A list of strings, where each dictionary must contain 
				the keys 'A' (Aspect), 'O' (Opinion), and 'S' (Sentiment).
		order: A string specifying the order of elements (e.g., 'aos', 'ao', 'as', 'a', 'o')

	Returns:
		A single string formatted with special tokens and elements based on the order
	"""
	global special_tokens
	# Convert order string to list of characters
	tuple_order = list(order.lower())
	
	# Build triplets based on the order length
	triplets = []
	for item in data_list:
		parts = []
		for element in tuple_order:
			if element.upper() in item:
				parts.append(f"{special_tokens[element]} {item[element.upper()].strip()}")
		triplets.append(" ".join(parts))
	
	# Join the list of strings together with a semicolon and space
	return ";".join(triplets)

def convert_input_to_legoabsa_format(input_str, order='aos'):
	global special_tokens
	global initial_definitions
	# Convert order string to list of characters
	tuple_order = list(order.lower())
	
	input_str = input_str.replace('[A] [O] [S]', '').strip()
	
	# Make replacements based on the order like this '{input_str}|aspect: <|box_start|> , opinion: <|quad_start|> , sentiment: <|vision_start|>'
	definitions = ' , '.join([f"{initial_definitions[element]}: {special_tokens[element]}" for element in tuple_order])
	return f"{input_str}| {definitions}"

In [129]:
convert_input_to_legoabsa_format("The room was clean but the service was terrible. The location is great though.", order='sa')

'The room was clean but the service was terrible. The location is great though.| sentiment: <|sentiment|> , aspect: <|aspect|>'

In [130]:
convert_output_to_legoabsa_format([{'A': 'harga', 'S': 'positive', 'O': 'terjangkau'},
 {'A': 'fasilitas', 'S': 'positive', 'O': 'nyaman'}], order='aos')

'<|aspect|> harga <|opinion|> terjangkau <|sentiment|> positive;<|aspect|> fasilitas <|opinion|> nyaman <|sentiment|> positive'

In [131]:
new_filtered_data = {}
for element_order_key, subtasks_orders in subtasks_orders_per_dataset.items():

	new_filtered_data[element_order_key] = {}
	for split in splits:
		new_filtered_data[element_order_key][split] = []
		data = deepcopy(dataset_per_split[split])
		
		print(f'Loaded {split} len {len(data)} element_order_key {element_order_key}')

		if split == 'train':
			used_order = deepcopy(subtasks_orders)
		else:
			used_order = [order for order in subtasks_orders if ''.join(order).lower() == original_order]

			print(f'Using only order {used_order} for {split} split since original order is {original_order}')

		# For each order, create a new dataset
		for order in used_order:
			new_data = []
			for instance in data:
				# print(f'---------------- Processing sentence_id {instance["sentence_id"]} | order {order} ----------------')
				parsed_data = parse_absa_string(instance['target'])
				new_parsed_data = []
				for triplet in parsed_data:
					new_triplet = {}
					for key in order:
						if key in triplet:
							new_triplet[key] = triplet[key]
					new_parsed_data.append(new_triplet)

				temp_data = deepcopy(new_parsed_data)
				# Remove all duplicate entries in new_parsed_data
				new_parsed_data = [dict(t) for t in {tuple(d.items()) for d in new_parsed_data}]
				if len(new_parsed_data) != len(temp_data):
					# # Calculate each frequency of entries in temp_data
					# frequency = {}
					# for d in temp_data:
					# 	key_tuple = tuple(d.items())
					# 	if key_tuple in frequency:
					# 		frequency[key_tuple] += 1
					# 	else:
					# 		frequency[key_tuple] = 1
					# duplicated_entries = {k: v for k, v in frequency.items() if v > 1}
					# print(f'Duplicates removed in sentence_id {instance["sentence_id"]} | order {order}:')
					# print('Before:', len(temp_data), temp_data)
					# # print('Before:', len(temp_data), instance['target'])
					# print('After:', len(new_parsed_data), new_parsed_data)
					# print('Removed entries:', duplicated_entries)
					pass
				
				# Remove instances with 'null' aspect
				temp_data = deepcopy(new_parsed_data)
				if ''.join(order).lower() in ['as', 'a']:
					new_parsed_data = [triplet for triplet in new_parsed_data if triplet.get('A', '').lower() != 'null']
					if len(new_parsed_data) != len(temp_data):
						# print(f'Null aspects removed in sentence_id {instance["sentence_id"]} | order {order}:')
						# print('Before:', len(temp_data), temp_data)
						# print('After:', len(new_parsed_data), new_parsed_data)
						pass
				
				# If no valid triplets, skip instance
				if len(new_parsed_data) == 0:
					# print(f'No valid triplets in sentence_id {instance["sentence_id"]} | order {order}, skipping instance.')
					# print(instance['input'])
					# print(parsed_data, instance['target'])
					continue

				# print(f'Final triplets')
				# print(instance['input'].replace('[A] [O] [S]', " ".join([f'[{key}]' for key in order])))
				# print(convert_to_absa_format(new_parsed_data, order=order))

				# Append to new_data
				new_data.append({
					'sentence_id': instance['sentence_id'],
					'instance_id': instance['instance_id'],
					'input': instance['input'].replace('[A] [O] [S]', " ".join([f'[{key}]' for key in order])),
					'target': convert_to_absa_format(new_parsed_data, order=order),
					'element_order': ''.join(order).lower(),
					'task_elements': ''.join(order).lower(),
					'dataset_type': instance['dataset_type'],
				})

			# Append new_data to new_filtered_data
			new_filtered_data[element_order_key][split].extend(new_data)
			
			print(f'------------- After filtering | order {"".join(order)} | {split} len {len(new_data)} -------------')

Loaded train len 4764 element_order_key indolegoabsa_multitask
------------- After filtering | order O | train len 4764 -------------
Loaded dev len 1284 element_order_key indolegoabsa_multitask
Using only order [['O']] for dev split since original order is o
------------- After filtering | order O | dev len 1284 -------------
Loaded test len 1286 element_order_key indolegoabsa_multitask
Using only order [['O']] for test split since original order is o
------------- After filtering | order O | test len 1286 -------------


In [132]:
for element_order_key in element_order_dict.keys():
    print(new_filtered_data[element_order_key]['train'].__len__(), new_filtered_data[element_order_key]['dev'].__len__(), new_filtered_data[element_order_key]['test'].__len__())

4764 1284 1286


In [133]:
for element_order_key, _ in element_order_dict.items():
	lego_absa_data_per_split = {}
	instance_id_counter = 0
	for split in splits:
		lego_absa_data_per_split[split] = []
		for instance in new_filtered_data[element_order_key][split]:
			absa_string = instance['target']
			parsed_absa = parse_absa_string(absa_string)
			try:
				listed_order = list(instance['element_order'].upper())
				for d in parsed_absa:
					assert all(key in d.keys() for key in listed_order)
			except:
				print("Error in instance:", instance)
				raise ValueError("Parsed ABSA contains invalid keys.")
			lego_input = convert_input_to_legoabsa_format(instance['input'], order=instance['element_order'])
			lego_output = convert_output_to_legoabsa_format(parsed_absa, order=instance['element_order'])
			lego_absa_data_per_split[split].append({
				"sentence_id": instance['sentence_id'],
				"instance_id": instance_id_counter,
				"input": lego_input,
				"target": lego_output,
				"element_order": instance['element_order'],
				'task_elements': instance['task_elements']
			})
			instance_id_counter += 1
	
	print(lego_absa_data_per_split['train'].__len__(), lego_absa_data_per_split['dev'].__len__(), lego_absa_data_per_split['test'].__len__())

	# Save the Lego-ABSA formatted dataset
	for split in splits:
		legoabsa_dataset_path = f'dataset/{dataset_type}/{lang}/{element_order_key}/{split}.json'
		os.makedirs(os.path.dirname(legoabsa_dataset_path), exist_ok=True)
		with open(legoabsa_dataset_path, 'w') as f:
			json.dump(lego_absa_data_per_split[split], f, ensure_ascii=False, indent=4)
		print(f"Lego-ABSA formatted dataset saved to {legoabsa_dataset_path}")


4764 1284 1286
Lego-ABSA formatted dataset saved to dataset/hoasa_hotel_o/indo/indolegoabsa_multitask/train.json
Lego-ABSA formatted dataset saved to dataset/hoasa_hotel_o/indo/indolegoabsa_multitask/dev.json
Lego-ABSA formatted dataset saved to dataset/hoasa_hotel_o/indo/indolegoabsa_multitask/test.json
